# D10：识别问题，保留证据，输出清洗结果

日期：2026-10-22。内核选择 Python (D Learning)，从上往下运行。
Shift + Enter 运行单元格；Ctrl + S 保存。
样本为虚构教学数据。请先跟练，再在 practice.ipynb 从空白实现。

目标：区分空价格、非数字价格、非法日期、完全重复和同资产同日期价格冲突。
样本 13 行：3 行无效、1 行多余副本、2 行价格冲突、7 行清洗结果。


## 截图代码：按国家统计包含 Python 的记录数量

country_grp 通常由 survey.groupby("Country") 得到。grp 是 group（分组）的简写。
选 LanguageWorkedWith 列后，apply 会把函数用于每个国家的这一列数据。
lambda x: ... 是匿名函数；这里 x 是一个国家内的 Series，不是单个字符串。
x.str.contains("Python") 给每条记录一个布尔判断，sum 将 True 按 1 累加。
统计的是包含这段文字的记录条数，并不是一段文字里 Python 出现的次数。

以下示例用 na=False 将缺失语言视为不匹配，用 regex=False 按普通文字查找。
若每条记录对应一个人，结果也就对应人数。截图 Length:179 表示结果有179个国家分组。


In [ ]:
import pandas as pd  # 导入 Pandas

survey = pd.DataFrame({  # 自拟五条记录，用于解释截图，不是图片中的原始调查数据
    "Country": ["China", "China", "China", "UK", "UK"],
    "LanguageWorkedWith": ["Python;SQL", "Java", "Python;C++", None, "JavaScript"],
})
country_grp = survey.groupby("Country")  # 按国家分组
python_counts = country_grp["LanguageWorkedWith"].apply(
    lambda x: x.str.contains("Python", na=False, regex=False).sum()
)
print(python_counts)  # China 为 2，UK 为 0


可以把匿名函数改写成具名函数：

```python
def count_python(x):
    matches = x.str.contains("Python", na=False, regex=False)
    return matches.sum()

python_counts = country_grp["LanguageWorkedWith"].apply(count_python)
```

今天后面的价格冲突检查也使用“按字段分组，再检查每组”的思路。


## 1. 在副本中制造异常

读取原始八行数据，先 copy，再向副本追加异常记录。original 始终保留原来的八行。
dtype="string" 先保留所有字段的原始文字；keep_default_na=False 让空字段保持空字符串，
方便区分“原本就是空的”和“非空但无法转成数字”。
.loc[len(raw)] 在本例连续默认索引后追加一行；concat 将完整重复副本接到表后，
ignore_index=True 生成连续默认索引。这里还没有开始清洗。


In [ ]:
from pathlib import Path  # Path：文件路径
import pandas as pd  # pandas 简写为 pd

data_dir = Path("D:/Learning/study/d10/data")  # 输入样本目录
output_dir = Path("D:/Learning/study/d10/outputs")  # 输出目录
output_dir.mkdir(parents=True, exist_ok=True)  # 创建输出目录，已有则继续

original = pd.read_csv(
    data_dir / "prices_original.csv",
    dtype="string",  # 先把原始内容读成文字
    keep_default_na=False,  # 空字段保持空字符串
)
original_columns = original.columns.tolist()  # 原始五列，后面用于判断完全重复
raw = original.copy()  # 所有异常只加入副本

raw.loc[len(raw)] = ["109", "2026-10-16", "A", "", "1400"]  # 空价格
raw.loc[len(raw)] = ["110", "2026-10-16", "B", "not-a-number", "1000"]  # 非数字价格
raw.loc[len(raw)] = ["111", "2026-02-30", "A", "109", "1500"]  # 非法日期
raw = pd.concat([raw, original.iloc[[0]].copy()], ignore_index=True)  # 完全复制 101
raw.loc[len(raw)] = ["112", "2026-10-13", "A", "999", "1200"]  # 与 103 价格冲突
raw = raw.astype("string")  # 统一保持原始文字类型

provided_dirty = pd.read_csv(
    data_dir / "prices_dirty.csv", dtype="string", keep_default_na=False
)
print("原样本形状：", original.shape)  # (8, 5)
print("异常副本形状：", raw.shape)  # (13, 5)
print("与已准备的脏样本一致：", raw.equals(provided_dirty))  # True
print(raw.tail(5))  # 观察新增的五行


## 2. 保留原始证据，新增转换后的字段

record_id=101 出现两次，所以它不能唯一定位本次输入的每一行。
source_row 表示 CSV 数据行序号，从 1 开始，不含表头；本例没有字段内换行。
原 date、price 都保留；只在 date_clean、price_clean 中保存转换结果。
转换失败的数字在当前可空类型下可能显示 <NA>，日期失败显示 NaT，统一用 isna 判断。


In [ ]:
work = raw.copy()  # 在新的工作副本中转换，不改原始文字表
work["source_row"] = range(1, len(work) + 1)  # 输入数据行序号 1～13

work["price_clean"] = pd.to_numeric(
    work["price"].str.strip(), errors="coerce"  # 无法转换的内容变成缺失值
)
work["date_clean"] = pd.to_datetime(
    work["date"].str.strip(), format="%Y-%m-%d", errors="coerce"
)  # 非法日期变成 NaT

print(work[["source_row", "record_id", "date", "date_clean", "price", "price_clean"]])
print(work[["price_clean", "date_clean"]].isna().sum())  # 价格缺失 2，日期缺失 1


## 3. 区分空价格、非数字价格与非法日期

不能只看转换后的缺失值，否则空字符串和 not-a-number 都会混在一起。
用保留的原始 price 区分二者，并把所有问题说明写入 issue。
问题说明采用拼接：同一行若同时有几类问题，不会因后一次赋值丢掉前面的说明。
invalid 为待核查的无效行；valid 为暂时通过价格和日期检查的行。


In [ ]:
empty_price = work["price"].str.strip().eq("")  # 原始价格为空；eq 表示是否相等
bad_number = work["price_clean"].isna() & ~empty_price  # 非空，但转不成数字
bad_date = work["date_clean"].isna()  # 日期无法解析

work["issue"] = ""  # issue：问题说明
for problem_mask, reason in [  # 逐类给满足条件的行补充说明
    (empty_price, "空价格；"),
    (bad_number, "非数字价格；"),
    (bad_date, "非法日期；"),
]:
    work.loc[problem_mask, "issue"] = work.loc[problem_mask, "issue"] + reason

invalid_mask = empty_price | bad_number | bad_date  # 任一项有问题就隔离
invalid = work.loc[invalid_mask].copy()  # 三行，编号 109、110、111
valid = work.loc[~invalid_mask].copy()  # 其余十行继续检查

print(invalid[["source_row", "record_id", "date", "price", "issue"]])
print("无效记录数：", len(invalid))  # 3
print("继续检查的行数：", len(valid))  # 10


## 4. 完全重复：识别整组，移除多余副本

本练习把原始五列全部相同定义为“完全重复”。source_row 和转换后的字段不参与比较。
keep=False 标记重复组的所有成员；keep="first" 只标记第一条以外的多余副本。
本例编号 101 的两个副本涉及两行，但只移除后出现的一行。


In [ ]:
duplicate_group_mask = valid.duplicated(subset=original_columns, keep=False)
extra_copy_mask = valid.duplicated(subset=original_columns, keep="first")

print("涉及完全重复的记录：")
print(valid.loc[duplicate_group_mask, ["source_row"] + original_columns])  # 数据行 1、12

duplicates = valid.loc[extra_copy_mask].copy()  # 只移除数据行 12 的副本
duplicates["issue"] = "完全重复副本：保留同内容第一条"
unique = valid.loc[~extra_copy_mask].copy()  # 剩余九行

print("移除的完全重复副本数：", len(duplicates))  # 1
print("去重后继续检查：", len(unique))  # 9


## 5. 同一资产同一天，检查有几种价格

同一个 asset、date_clean 组合是一个分组。nunique 表示不同值的数量。
transform("nunique") 把每组的不同价格数量对应回每个成员，得到与 unique 行索引相同的 Series。
数量大于 1 才表示价格冲突；仅仅日期和资产重复，不能直接推出价格冲突。
两条冲突记录一起另存待核实，本练习不会根据先后顺序或价格大小决定哪条正确。


In [ ]:
grouped_prices = unique.groupby(["asset", "date_clean"])["price_clean"]
price_count = grouped_prices.transform("nunique")  # 每一行对应组内不同价格的数量
conflict_mask = price_count > 1  # 一组有两种及以上价格，说明有冲突

conflicts = unique.loc[conflict_mask].copy()  # 编号 103、112，一起隔离
conflicts["issue"] = "同资产同日期价格冲突：待核实"
clean_work = unique.loc[~conflict_mask].copy()  # 剩余七行进入清洗结果

print(conflicts[["source_row", "record_id", "asset", "date_clean", "price_clean"]])
print("冲突记录数：", len(conflicts))  # 2
print("清洗结果行数：", len(clean_work))  # 7


## 6. 分别导出异常记录与清洗结果

异常文件保留原始日期、价格、转换值、来源行及问题说明。
清洗文件采用转换后的日期和价格，并保留来源行，便于回查。
index=False 不额外导出 Pandas 的行索引；source_row 已经作为普通列保存。
utf-8-sig 便于常见 Windows 表格软件识别中文问题说明。


In [ ]:
anomalies = pd.concat([invalid, duplicates, conflicts], ignore_index=True)
audit_columns = [  # 输出需要保留的证据字段
    "source_row", "record_id", "date", "asset", "price", "volume",
    "date_clean", "price_clean", "issue",
]
anomalies = anomalies.loc[:, audit_columns].sort_values("source_row").reset_index(drop=True)

clean = clean_work.loc[:, [
    "source_row", "record_id", "date_clean", "asset", "price_clean", "volume"
]].rename(columns={"date_clean": "date", "price_clean": "price"})
clean = clean.sort_values(["asset", "date"]).reset_index(drop=True)

anomalies.to_csv(output_dir / "anomalies.csv", index=False, encoding="utf-8-sig")
clean.to_csv(output_dir / "prices_clean.csv", index=False, encoding="utf-8-sig")

print("异常文件：", output_dir / "anomalies.csv")
print("清洗结果：", output_dir / "prices_clean.csv")
print("输入/异常/清洗：", len(raw), len(anomalies), len(clean))  # 13、6、7
assert len(raw) == len(anomalies) + len(clean)  # assert：核对每一行都有去向
print(clean)


## 7. 复做第 217 题并补充边界情况

先在空白练习中写 contains_duplicate(nums)，再运行下面的七组输入。
“所有值相同”的情况在第二个元素就能发现重复；“没有重复”需要检查全部元素。
集合查找、添加平均 O(1)，最多检查 n 个元素，所以整体平均 O(n)，额外空间最坏 O(n)。
空列表是本地拓展，不是本题输入约束内必须出现的情况。


In [ ]:
def contains_duplicate(nums):  # 判断是否存在重复值
    seen = set()  # 保存此前见过的数字
    for number in nums:  # 按顺序检查每个数字
        if number in seen:  # 先检查，不能先添加
            return True  # 已经出现过，立即返回
        seen.add(number)  # 第一次见到，记录下来
    return False  # 全部检查完仍无重复

examples = [
    ([2, 5, 2], True),  # 非相邻重复
    ([1, 2, 3, 4], False),  # 没有重复
    ([7, 7, 7, 7], True),  # 所有值相同
    ([7], False),  # 只有一个数字
    ([-1, 0, -1], True),  # 包含负数
    ([0, 0], True),  # 重复的零
    ([], False),  # 本地拓展：空列表
]
for nums, expected in examples:  # 分别取输入与预期结果
    result = contains_duplicate(nums)
    assert result is expected  # 检查结果与预期一致
    print(nums, "->", result)


## 验收

不看参考版，在 practice.ipynb 完成识别、分类、导出及行数核对，并回答 notes.md。
输出应为 anomalies.csv 六行、prices_clean.csv 七行，13 = 3 + 1 + 2 + 7。
说清为什么空价格与非数字价格要区分、为什么完全重复只移除多余副本、
为什么价格冲突两条一起待核实。
